## Connector demo
compatible with 
- EDC v0.10.1, 
- EDC v0.14.0

most examples based on the edc samples: https://github.com/eclipse-edc/Samples

In [ ]:
import requests
import json
import time
from typing import List, Dict
import re
from urllib.parse import unquote
from uuid import uuid4

from dataspace_apis import *

### Demo setup

In [ ]:

IS_LOCALHOST_DEPLOYMENT = False

PROVIDER_URL = "https://sage-connector-edc-connector.apps.bst2.paas.psnc.pl"
CONSUMER_URL = "https://consumer-connector-edc-connector.apps.bst2.paas.psnc.pl"
FEDERATED_CATALOG_BASE_URL = "https://federatedcatalog-edc-connector.apps.bst2.paas.psnc.pl"
CONSUMER_BACKEND_URL = "https://consback-edc-connector.apps.bst2.paas.psnc.pl"

# for local:
LOCALHOST = "http://localhost"
CONSUMER_CONTAINER = "http://consumer-connector"
PROVIDER_CONTAINER = "http://provider-connector"
FEDERATED_CATALOG_BASE_CONTAINER = "http://federated-catalog"
CONSUMER_BACKEND_CONTAINER = "http://consumer-backend"

if (IS_LOCALHOST_DEPLOYMENT):
    CONSUMER_URL = LOCALHOST
    PROVIDER_URL = LOCALHOST
    FEDERATED_CATALOG_BASE_URL = LOCALHOST

In [ ]:
PROVIDER_API = f"{PROVIDER_URL}/api"
PROVIDER_CONTROL = f"{PROVIDER_URL}/api/control"
PROVIDER_MANAGEMENT = f"{PROVIDER_URL}/api/management"
PROVIDER_PROTOCOL = f"{PROVIDER_URL}/api/dsp"
PROVIDER_PUBLIC = f"{PROVIDER_URL}/api/public"

CONSUMER_API = f"{CONSUMER_URL}/api"
CONSUMER_CONTROL = f"{CONSUMER_URL}/api/control"
CONSUMER_MANAGEMENT = f"{CONSUMER_URL}/api/management"
CONSUMER_PROTOCOL = f"{CONSUMER_URL}/api/dsp"
CONSUMER_PUBLIC = f"{CONSUMER_URL}/api/public"

CONSUMER_BACKEND_EDR = f"{CONSUMER_BACKEND_URL}/edr-endpoint"
FEDERATED_CATALOG_URL = f"{FEDERATED_CATALOG_BASE_URL}/api/catalog"

if (IS_LOCALHOST_DEPLOYMENT):
    PROVIDER_API = f"{PROVIDER_URL}:8190/api"
    PROVIDER_CONTROL = f"{PROVIDER_URL}:8193/api/control"
    PROVIDER_MANAGEMENT = f"{PROVIDER_URL}:8191/api/management"
    PROVIDER_PROTOCOL = f"{PROVIDER_URL}:8192/api/dsp"
    PROVIDER_PUBLIC = f"{PROVIDER_URL}:12001/api/public"

    CONSUMER_API = f"{CONSUMER_URL}:8080/api"
    CONSUMER_CONTROL = f"{CONSUMER_URL}:8083/api/control"
    CONSUMER_MANAGEMENT = f"{CONSUMER_URL}:8081/api/management"
    CONSUMER_PROTOCOL = f"{CONSUMER_URL}:8082/api/dsp"
    CONSUMER_PUBLIC = f"{CONSUMER_URL}:11001/api/public"

    CONSUMER_BACKEND_EDR = f"{CONSUMER_BACKEND_CONTAINER}:4000/edr-endpoint"
    FEDERATED_CATALOG_URL = f"{FEDERATED_CATALOG_BASE_URL}:8294/api/catalog"

In [ ]:
"""
The vars below are useful when working on local deployment
(or at least when no routings are specified for connectors)
Docker containers have their own localhost, which is not the host
machine's localhost.

In some requests there is a 'counterPartyAddress' which contains
localhost. This one will be solved as containers' internal localhost
but not the localhost of the host machine and connectors won't connect
to each other.

Hence we substitute the "localhost" with containers' names (if they
contain "localhost", otherwise urls remain unchanged).
If routings are specified correctly on PaaS, those lines aren't
required.

We leave those vars here anyway, just not to complicate 
any of the requests later in the demo. 
"""

provider_control_internal = PROVIDER_CONTROL.replace(LOCALHOST, PROVIDER_CONTAINER)
provider_public_internal = PROVIDER_PUBLIC.replace(LOCALHOST, PROVIDER_CONTAINER)
provider_protocol_internal = f"{PROVIDER_PROTOCOL}".replace(LOCALHOST, PROVIDER_CONTAINER)

default_headers = {
    "Content-Type": "application/json",
    "x-api-key": "password",
}

### Conn Check

In [ ]:
def check_health(coreApiUrl):
    print(f"{coreApiUrl}/check/health/")
    rp = requests.get(f"{coreApiUrl}/check/health/", headers=default_headers).json()
    print(rp)

check_health(PROVIDER_API)
check_health(CONSUMER_API)
print("They're Alive!")

### Content reader script 

In [ ]:
def extract_name_from_uri(uri: str) -> str:
    """
    Extracts the name from a URI.
    Assumes the name is the last part of the URI after the last slash.
    """
    name = uri.split('/')[-1]
    _hash = ''
    if '?' in name:
        res = re.split(r'\?|\%3F', name, maxsplit=1)
        if len(res) == 1:
            name = res[0]
        elif len(res) == 2:
            name, _hash = res
        _hash = str(hash(_hash))
    return f"{name}{"_" + _hash if _hash != "" else ""}"

In [ ]:
def extract_content_type_from_uri(uri: str) -> str:
    """
    Extracts the content type from a URI.
    Assumes the content type is the file extension of the last part of the URI.
    """
    name = uri.split('/')[-1]
    if '.' in name:
        return name.split('.')[-1]
    return "json"

In [ ]:
path = './dcat-dump.jsonld'
URL = 'dcat:landingPage'
IDENTIFIER = 'dct:identifier'
TITLE = 'dct:title'
DESCRIPTION = 'dct:description'
SPATIAL = 'http://purl.org/dc/terms/spatial'

records : Dict = {}

with open(path, 'r') as f:
    records = json.load(f)

policy_id = "test-policy"
policy = create_policy(policy_id, PROVIDER_MANAGEMENT, default_headers, permissions=[])
print(f'Policy uploaded with status code: {policy.status_code}')

print(records["@context"])
for record in records["dcat:dataset"]:
    asset_id = record[IDENTIFIER] or uuid4()
    contract_definition_id = f"cd_{asset_id}"
    url = record[URL]["@id"] if URL in record else "https://jsonplaceholder.typicode.com/users"
    content_type = extract_content_type_from_uri(url)
    asset = create_asset( # update
        asset_id=asset_id,
        management_url=PROVIDER_MANAGEMENT,
        default_headers=default_headers,
        asset_name=record[TITLE],
        content_type=content_type,
        baseUrl=record[URL]["@id"] if URL in record else "https://jsonplaceholder.typicode.com/users",
        additional_metadata={k: v for k, v in record.items() if k not in [IDENTIFIER, TITLE, URL]},
        proxy=True
    )
    print(f"Asset uploaded with status code: {asset.status_code}")
    contract_definition = create_contract_definition(contract_definition_id, PROVIDER_MANAGEMENT, asset_id, policy_id, default_headers)
    print(f"Contract definition created with status code: {contract_definition.status_code}")

    # rm_asset = delete_asset(asset_id, PROVIDER_MANAGEMENT, default_headers)
    # print(f'Asset removed with code: {rm_asset.status_code}')
    # rm_cdef = delete_contract_definition(contract_definition_id, PROVIDER_MANAGEMENT, default_headers)
    # print(f'Contract definition removed with code: {rm_cdef.status_code}')

    # break
    time.sleep(0.01)
    